In [1]:
# ==========================================
# PREPARACIÓN DEL ENTORNO
# ==========================================
using Pkg
Pkg.activate("/home/antonibancells/Desktop/Codi/TFM/ProvesModernes")

using ITensors
using ITensorMPS
using Plots
using Printf

  Activating project at `~/Desktop/Codi/TFM/ProvesModernes`


El estado cuántico $|\psi\rangle$ se implementa mediante un objeto ITensor. Sus índices se definen mediante siteinds de tipo "Qubit". Para introducir valores en el tensor, es necesario realizar una codificación binaria, que puede hacerse de dos maneras:
<ul>
    <li>MSB en la posición 1</li>
    <li>LSB en la posición 1</li>
</ul>

Para el cálculo de la derivada se necesita la implementación de los operadors de desplazamiento, ya sea $F|i\rangle=|i+1\rangle$ (FSB( o $B|i\rangle=|i-1\rangle$ (BSB)

El primer caso consiste en aplicar $E|i\rangle=|i+1\rangle$ con MSB en la posición 1. En este caso, el operador derivada queda expresado como $$D^{(1)}=\frac{E^\dagger-E}{2dx}$$

In [2]:
function construir_derivada_mpo(sites, dx)
    #Obtención del número de qubits en formato QTT
    N = length(sites)
    # Creación del MPO de desplazamiento FSB
    # Inicialización del tensor formado por un vector de N ITensors que contendrá al MPO, estan inicializados a 0
    tensors_E = Vector{ITensor}(undef, N)
    # Creación de los índices de enlace ("links" o "bonds") virtuales entre los tensores del MPO
    # Tienen dimensión 2 porque únicamente necesitan transmitir 2 estados: 1 (no hay carry) o 2 (si hay carry)
    links_E = [Index(2, "LinkE,l=$i") for i in 1:(N-1)]

    # BUCLE PRINCIPAL: CONSTRUCCIÓN CÓDIGO DE SUMA BINARIA +1 (CARRY PROPAGATION)
    # Recorremos los tensors de DERECHA a IZQUIERDA (del bit menos significativo LSB al más significativo MSB
    for j in N:-1:1
        s = sites[j] #Índice físico original
        sp = s'      #Índice físico "prima", con el estado final despues de la operación
        # CASO A: El bit menos significativo (LSB) - Lugar donde se inyecta la suma de +1
        if j == N  # LSB: Aquí comença la suma de +1 
            # Se crea un tensor auxiliar temporal
            # El tensor tiene 3 índices: los dos físics (s, sp) y ellink izquierdo (j-1) para pasar el carry.
            T = ITensor(s, sp, links_E[j-1])
            # Si el bit és |0> (estado 1 en ITensor), al sumarle 1 passa a ser |1> (estado 2). No genera carry.
            T[s=>1, sp=>2, links_E[j-1]=>1] = 1.0  # |0> -> |1>, carry=0
            # Si el bit és |1> (estado 2 en ITensor), al sumarle 1 es converteix en |0> (estado 1).
            # Genera un carry que se propaga hacia la izquierda (estado 2 del link).
            T[s=>2, sp=>1, links_E[j-1]=>2] = 1.0  # |1> -> |0>, carry=1
            #Se guarda el tensor creado en la posición j=N del vector de tensores tensors_E
            tensors_E[j] = T
        # CASO B: El bit más significativo (MSB) - Final de la propagación del carry
        elseif j == 1  # MSB: Final de la propagación
            T = ITensor(links_E[j], s, sp)
            # Si el link de la derecha dice "no viene carry" (estat 1): el qubit no canvia.
            T[links_E[j]=>1, s=>1, sp=>1] = 1.0    # |0> es queda en |0>
            T[links_E[j]=>1, s=>2, sp=>2] = 1.0    # |1> es queda en |1>
            # Si el link de la derecha dice "viene carry" (estat 2): sumamos 1 a este bit.         
            T[links_E[j]=>2, s=>1, sp=>2] = 1.0    # Ve carry -> suma 1
            T[links_E[j]=>2, s=>2, sp=>1] = 1.0    # Periòdic (torna a 0)
            tensors_E[j] = T
        # CAS C: Tensores intermedios del
        else  # Tensors intermedis (CORREGIT: tot utilitza links_E)
            T = ITensor(links_E[j], s, sp, links_E[j-1])
            # Bloque 1: No viene carry de la derecha (link_E[j]=>1). No cambia nada, pasamos carry=0 (link_E[j-1]=>1).
            T[links_E[j]=>1, s=>1, sp=>1, links_E[j-1]=>1] = 1.0
            T[links_E[j]=>1, s=>2, sp=>2, links_E[j-1]=>1] = 1.0
            # Bloque 2: Sí que viene carry de la derecha (link_E[j]=>2).
            # Si el qubit és |0>, es transforma en |1> i el carry se absorve (se envía carry=0).
            T[links_E[j]=>2, s=>1, sp=>2, links_E[j-1]=>1] = 1.0
            # Si el qubit és |1>, es transforma en |0> i el carry es manté actiu (enviem carry=1) (link_E[j-1]=>2).
            T[links_E[j]=>2, s=>2, sp=>1, links_E[j-1]=>2] = 1.0
            tensors_E[j] = T
        end
    end
    E = MPO(tensors_E)
    
    # CORREGIT: utilitzem conj(E) i girem els primes per obtenir l'adjunt (shift esquerra)
    E_dag = swapprime(conj(E), 0 => 1)
    
    # Ajuntem-ho tot directament a nivell d'MPO: D = (E - E†) / (2*dx)
    D_mpo = add(1.0 / (2.0 * dx) * E_dag, -1.0 / (2.0 * dx) * E; cutoff=1e-12)
    return D_mpo
end

construir_derivada_mpo (generic function with 1 method)

In [3]:
function main()
    N = 10                  
    dim_total = 1 << N      
    x_min = 0.0
    x_max = 2 * π           
    dx = (x_max - x_min) / dim_total 

    # Creacion de los sitios para N qubits
    sites = siteinds("Qubit", N)

    # Creación del tensor que contendrá los valores de la función
    psi_tensor = ITensor(sites...)
    
    # Introducción de los valores de la función para cada configuración de los N índices
    for i in 0:(dim_total - 1)
        # Coordenada x de la función a calcular
        x = x_min + i * dx
        # Valor de la función para esta coordenada
        val = sin(x)
        # Array binario con los bits, donde el elemento 1 contiene el MSB
        estado_binario= [((i >> (N - j)) & 1) + 1 for j in 1:N]
        psi_tensor[estado_binario...] = val
    end
    
    # Creación del MPS a partir del ITensor creado previamente. Aquí se produce la compresión
    psi = MPS(psi_tensor, sites; cutoff=1e-12, maxdim=10)

    # Llamada a la función constructora del MPO de la derivada
    D_mpo = construir_derivada_mpo(sites, dx)
    println("-> MPO de la derivada generado con éxito. Rango: ", maxlinkdim(D_mpo))

    # Aplicación del MPO sobre el MPS inicial
    d_psi = contract(D_mpo, psi; cutoff=1e-12)
    d_psi = noprime(d_psi)

    # Reconstrucción del tensor procesado a partir del MPS
    res_tensor = d_psi[1]
    for j in 2:N res_tensor *= d_psi[j] end

    println("\nComprovación de los resultados:")
        
    for i in 0:100:1000
        x = x_min + i * dx
        estado_binario = [((i >> (N - j)) & 1) + 1 for j in 1:N]
        val_qtt = res_tensor[estado_binario...]
        @printf("x = %7.4f | QTT: %7.4f | Real : %7.4f | Error: %7.4e\n", 
                x, val_qtt, cos(x), abs(val_qtt - cos(x)))
    end
end

main()

-> MPO de la derivada generado con éxito. Rango: 4

Comprovación de los resultados:
x =  0.0000 | QTT:  1.0000 | Real :  1.0000 | Error: 6.2749e-06
x =  0.6136 | QTT:  0.8176 | Real :  0.8176 | Error: 5.1303e-06
x =  1.2272 | QTT:  0.3369 | Real :  0.3369 | Error: 2.1140e-06
x =  1.8408 | QTT: -0.2667 | Real : -0.2667 | Error: 1.6736e-06
x =  2.4544 | QTT: -0.7730 | Real : -0.7730 | Error: 4.8506e-06
x =  3.0680 | QTT: -0.9973 | Real : -0.9973 | Error: 6.2579e-06
x =  3.6816 | QTT: -0.8577 | Real : -0.8577 | Error: 5.3822e-06
x =  4.2951 | QTT: -0.4052 | Real : -0.4052 | Error: 2.5429e-06
x =  4.9087 | QTT:  0.1951 | Real :  0.1951 | Error: 1.2241e-06
x =  5.5223 | QTT:  0.7242 | Real :  0.7242 | Error: 4.5446e-06
x =  6.1359 | QTT:  0.9892 | Real :  0.9892 | Error: 6.2070e-06


In [4]:
function construir_derivada_mpo(sites, dx)
    N = length(sites)
    tensors_E = Vector{ITensor}(undef, N)
    links_E = [Index(2, "LinkE,l=$i") for i in 1:(N-1)]
    
    # BUCLE D'ESQUERRA A DRETA (Alineat amb j-1)
    for j in 1:N
        s = sites[j]
        sp = s'
        
        if j == 1  # LSB (Ara està a la posició 1, aquí neix la suma!)
            T = ITensor(s, sp, links_E[1])
            T[s=>1, sp=>2, links_E[1]=>1] = 1.0  # |0> -> |1>, carry=0
            T[s=>2, sp=>1, links_E[1]=>2] = 1.0  # |1> -> |0>, carry=1
            tensors_E[j] = T
        elseif j == N  # MSB (Ara està a la posició N, final de la suma)
            T = ITensor(links_E[j-1], s, sp)
            T[links_E[j-1]=>1, s=>1, sp=>1] = 1.0
            T[links_E[j-1]=>1, s=>2, sp=>2] = 1.0
            T[links_E[j-1]=>2, s=>1, sp=>2] = 1.0
            T[links_E[j-1]=>2, s=>2, sp=>1] = 1.0
            tensors_E[j] = T
        else  # Tensors intermedis
            T = ITensor(links_E[j-1], s, sp, links_E[j])
            T[links_E[j-1]=>1, s=>1, sp=>1, links_E[j]=>1] = 1.0
            T[links_E[j-1]=>1, s=>2, sp=>2, links_E[j]=>1] = 1.0
            T[links_E[j-1]=>2, s=>1, sp=>2, links_E[j]=>1] = 1.0
            T[links_E[j-1]=>2, s=>2, sp=>1, links_E[j]=>2] = 1.0
            tensors_E[j] = T
        end
    end
    E = MPO(tensors_E)
    E_dag = swapprime(conj(E), 0 => 1)
    
    # I a sobre, amb aquesta combinació, la fórmula teòrica és directa, sense girs de signe!
    D_mpo = add(1.0 / (2.0 * dx) * E_dag, -1.0 / (2.0 * dx) * E; cutoff=1e-12)
    return D_mpo
end

construir_derivada_mpo (generic function with 1 method)

In [5]:
function main()
    N = 10                  
    dim_total = 1 << N      
    x_min = 0.0
    x_max = 2 * π           
    dx = (x_max - x_min) / dim_total 

    # Creacion de los sitios para N qubits
    sites = siteinds("Qubit", N)

    # Creación del tensor que contendrá los valores de la función
    psi_tensor = ITensor(sites...)

    # Introducción de los valores de la función para cada configuración de los N índices
    for i in 0:(dim_total - 1)
        # Coordenada x de la función a calcular
        x = x_min + i * dx
        # Valor de la función para esta coordenada
        val = sin(x)
        # Array binario con los bits, donde el elemento 1 contiene el MSB
        estado_binario= [((i >> (j - 1)) & 1) + 1 for j in 1:N]
        psi_tensor[estado_binario...] = val
    end
    
    # Creación del MPS a partir del ITensor creado previamente. Aquí se produce la compresión    
    psi = MPS(psi_tensor, sites; cutoff=1e-12, maxdim=10)

    # Llamada a la función constructora del MPO de la derivada
    D_mpo = construir_derivada_mpo(sites, dx)
    println("-> MPO de la derivada generado con éxito. Rango: ", maxlinkdim(D_mpo))

    # Aplicación del MPO sobre el MPS inicial
    d_psi = contract(D_mpo, psi; cutoff=1e-12) # Se realiza compresión SVD adicional
    d_psi = noprime(d_psi)

    # Reconstrucción del tensor procesado a partir del MPS
    res_tensor = d_psi[1]
    for j in 2:N res_tensor *= d_psi[j] end
    
    println("\nComprovación de los resultados:")

    for i in 0:100:1000
        x = x_min + i * dx
        estado_binario = [((i >> (j - 1)) & 1) + 1 for j in 1:N]
        val_qtt = res_tensor[estado_binario...]
        @printf("x = %7.4f | QTT: %7.4f | Real : %7.4f | Error: %7.4e\n", 
                x, val_qtt, cos(x), abs(val_qtt - cos(x)))
    end
end

main()

-> MPO de la derivada generado con éxito. Rango: 3

Comprovación de los resultados:
x =  0.0000 | QTT:  1.0000 | Real :  1.0000 | Error: 6.2749e-06
x =  0.6136 | QTT:  0.8176 | Real :  0.8176 | Error: 5.1303e-06
x =  1.2272 | QTT:  0.3369 | Real :  0.3369 | Error: 2.1140e-06
x =  1.8408 | QTT: -0.2667 | Real : -0.2667 | Error: 1.6736e-06
x =  2.4544 | QTT: -0.7730 | Real : -0.7730 | Error: 4.8506e-06
x =  3.0680 | QTT: -0.9973 | Real : -0.9973 | Error: 6.2579e-06
x =  3.6816 | QTT: -0.8577 | Real : -0.8577 | Error: 5.3822e-06
x =  4.2951 | QTT: -0.4052 | Real : -0.4052 | Error: 2.5429e-06
x =  4.9087 | QTT:  0.1951 | Real :  0.1951 | Error: 1.2242e-06
x =  5.5223 | QTT:  0.7242 | Real :  0.7242 | Error: 4.5446e-06
x =  6.1359 | QTT:  0.9892 | Real :  0.9892 | Error: 6.2070e-06


In [6]:
function construir_derivada_mpo_estandar_corregido(sites, dx)
    N = length(sites)
    tensors_E = Vector{ITensor}(undef, N)
    
    # links_E[j] conectará el sitio j con el j+1
    links_E = [Index(2, "LinkE,l=$i") for i in 1:(N-1)]
    
    # AHORA: El sitio 1 es el LSB (aquí empieza la resta)
    # El acarreo (deuda) viaja de izquierda a derecha (de 1 a N)
    for j in 1:N
        s = sites[j]
        sp = s'
        
        # ---------------------------------------------------------------------
        # CASO A: LSB (Sitio 1) - Aquí se inicia la resta de -1
        # ---------------------------------------------------------------------
        if j == 1  
            # Conecta con el sitio 2 a través de links_E[1]
            T = ITensor(s, sp, links_E[j])
            
            # |1> (físico 2) - 1 = |0> (físico 1). No hay deuda.
            T[s=>2, sp=>1, links_E[j]=>1] = 1.0  
            # |0> (físico 1) - 1 = |1> (físico 2). Se genera deuda (estado 2).
            T[s=>1, sp=>2, links_E[j]=>2] = 1.0  
            
            tensors_E[j] = T
            
        # ---------------------------------------------------------------------
        # CASO B: MSB (Sitio N) - Final de la propagación de deuda
        # ---------------------------------------------------------------------
        elseif j == N  
            # Viene del sitio N-1 a través de links_E[N-1]
            T = ITensor(links_E[j-1], s, sp)
            
            # Si NO viene deuda (link => 1), se queda igual
            T[links_E[j-1]=>1, s=>1, sp=>1] = 1.0
            T[links_E[j-1]=>1, s=>2, sp=>2] = 1.0
            
            # Si SÍ viene deuda (link => 2), restamos 1
            T[links_E[j-1]=>2, s=>2, sp=>1] = 1.0  # |1> -> |0>
            T[links_E[j-1]=>2, s=>1, sp=>2] = 1.0  # |0> -> |1> (Periódico)
            
            tensors_E[j] = T
            
        # ---------------------------------------------------------------------
        # CASO C: Sitios intermedios
        # ---------------------------------------------------------------------
        else  
            # links_E[j-1] viene de la izquierda, links_E[j] va hacia la derecha
            T = ITensor(links_E[j-1], s, sp, links_E[j])
            
            # Si NO viene deuda
            T[links_E[j-1]=>1, s=>1, sp=>1, links_E[j]=>1] = 1.0
            T[links_E[j-1]=>1, s=>2, sp=>2, links_E[j]=>1] = 1.0
            
            # Si SÍ viene deuda
            T[links_E[j-1]=>2, s=>2, sp=>1, links_E[j]=>1] = 1.0 # Deuda resuelta
            T[links_E[j-1]=>2, s=>1, sp=>2, links_E[j]=>2] = 1.0 # Deuda se propaga
            
            tensors_E[j] = T
        end
    end
    
    E = MPO(tensors_E)
    E_dag = swapprime(conj(E), 0 => 1)
    
    # Operador derivada
    D_mpo = add((1.0 / (2.0 * dx)) * E, (-1.0 / (2.0 * dx)) * E_dag; cutoff=1e-12)
    
    return D_mpo
end

function main()
    N = 10                  
    dim_total = 1 << N      
    x_min = 0.0
    x_max = 2 * π           
    dx = (x_max - x_min) / dim_total 

    sites = siteinds("Qubit", N)

    psi_tensor = ITensor(sites...)
    
    # Rellenamos el tensor mapeando: Sitio 1 = LSB, Sitio N = MSB
    for i in 0:(dim_total - 1)
        x = x_min + i * dx
        val = sin(x)
        
        # CAMBIO AQUÍ: j=1 es el LSB (i >> 0), j=N es el MSB (i >> N-1)
        estado_binario = [((i >> (j - 1)) & 1) + 1 for j in 1:N]
        psi_tensor[estado_binario...] = val
    end
    
    psi = MPS(psi_tensor, sites; cutoff=1e-12, maxdim=64)

    # Llamamos a nuestro nuevo MPO corregido
    D_mpo = construir_derivada_mpo_estandar_corregido(sites, dx)
    println("-> MPO de la derivada generado con éxito. Rango: ", maxlinkdim(D_mpo))

    # Aplicamos el MPO
    d_psi = contract(D_mpo, psi; cutoff=1e-12)
    d_psi = noprime(d_psi)

    # Reconstruimos el tensor para verificar
    res_tensor = d_psi[1]
    for j in 2:N 
        res_tensor *= d_psi[j] 
    end

    println("\nComprobación de los resultados:")
        
    for i in 0:100:1000
        x = x_min + i * dx
        # Usamos exactamente el mismo mapeo de bits para leer el resultado
        estado_binario = [((i >> (j - 1)) & 1) + 1 for j in 1:N]
        val_qtt = res_tensor[estado_binario...]
        @printf("x = %7.4f | QTT: %7.4f | Real : %7.4f | Error: %7.4e\n", 
                x, val_qtt, cos(x), abs(val_qtt - cos(x)))
    end
end

main()

-> MPO de la derivada generado con éxito. Rango: 3

Comprobación de los resultados:
x =  0.0000 | QTT:  1.0000 | Real :  1.0000 | Error: 6.2749e-06
x =  0.6136 | QTT:  0.8176 | Real :  0.8176 | Error: 5.1303e-06
x =  1.2272 | QTT:  0.3369 | Real :  0.3369 | Error: 2.1140e-06
x =  1.8408 | QTT: -0.2667 | Real : -0.2667 | Error: 1.6736e-06
x =  2.4544 | QTT: -0.7730 | Real : -0.7730 | Error: 4.8506e-06
x =  3.0680 | QTT: -0.9973 | Real : -0.9973 | Error: 6.2579e-06
x =  3.6816 | QTT: -0.8577 | Real : -0.8577 | Error: 5.3822e-06
x =  4.2951 | QTT: -0.4052 | Real : -0.4052 | Error: 2.5428e-06
x =  4.9087 | QTT:  0.1951 | Real :  0.1951 | Error: 1.2242e-06
x =  5.5223 | QTT:  0.7242 | Real :  0.7242 | Error: 4.5446e-06
x =  6.1359 | QTT:  0.9892 | Real :  0.9892 | Error: 6.2070e-06
